In [1]:
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
def calculate_ci(metric_value, n_samples):
    """Calculates the 95% confidence interval for a proportion."""
    if n_samples == 0:
        return 0.0
    # Ensure the metric value is a proportion (between 0 and 1)
    clamped_value = np.clip(metric_value, 0, 1)
    sigma = np.sqrt(clamped_value * (1 - clamped_value) / n_samples)
    ci = 1.96 * sigma  # 95% confidence interval
    return ci

def plot_curves(log_path, output_dir, dataset_sizes):
    """
    Parses the log.txt file and creates targeted plots for specific metrics.
    """
    epochs_data = []
    try:
        with open(log_path, 'r') as f:
            for line in f:
                try:
                    epochs_data.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f"Skipping malformed line in log.txt: {line.strip()}")
    except IOError as e:
        print(f"Error reading log file: {e}")
        return

    if not epochs_data:
        print("Log file is empty. No curves to plot.")
        return
        
    epochs = [e['epoch'] for e in epochs_data]

    # Define the specific plots we want to create
    plot_configurations = [
        {
            "title": "Loss Curves",
            "ylabel": "Loss",
            "filename": "loss_curves.png",
            "metrics": {
                "train_loss": "Train Loss",
                "val_loss": "Validation Loss",
                "test_loss": "Test Loss"
            },
            "show_ci": False
        },
        {
            "title": "Accuracy Curves",
            "ylabel": "Balanced Accuracy",
            "filename": "accuracy_curves.png",
            "metrics": {
                "train_class_acc": "Train Accuracy",
                "val_balanced_accuracy": "Validation Balanced Accuracy",
                "test_balanced_accuracy": "Test Balanced Accuracy"
            },
            "show_ci": True
        },
        {
            "title": "ROC AUC Curves",
            "ylabel": "ROC AUC",
            "filename": "roc_auc_curves.png",
            "metrics": {
                "val_roc_auc": "Validation ROC AUC",
                "test_roc_auc": "Test ROC AUC"
            },
            "show_ci": True
        },
        {
            "title": "PR AUC Curves",
            "ylabel": "PR AUC",
            "filename": "pr_auc_curves.png",
            "metrics": {
                "val_pr_auc": "Validation PR AUC",
                "test_pr_auc": "Test PR AUC"
            },
            "show_ci": True
        }
    ]

    for config in plot_configurations:
        plt.figure(figsize=(10, 6))
        
        for key, label in config["metrics"].items():
            # Check if the metric exists in the log data
            if key in epochs_data[0]:
                values = np.array([e[key] for e in epochs_data if key in e])
                # Skip if all values are None or inf
                if np.all(np.isinf(values)) or np.all(np.isnan(values)):
                    continue

                plt.plot(epochs, values, marker='o', linestyle='-', label=label)

                # Add confidence intervals if requested and applicable
                if config["show_ci"]:
                    split = key.split('_')[0] # 'train', 'val', or 'test'
                    if split in dataset_sizes:
                        ci = calculate_ci(values, dataset_sizes[split])
                        plt.fill_between(epochs, values - ci, values + ci, alpha=0.2)

        plt.title(config["title"])
        plt.xlabel('Epoch')
        plt.ylabel(config["ylabel"])
        plt.grid(True)
        plt.legend()
        
        plot_filename = os.path.join(output_dir, config["filename"])
        plt.savefig(plot_filename)
        plt.close()
        print(f"Saved plot: {plot_filename}")

In [ ]:


if not os.path.isdir(args.output_dir):
    print(f"Error: Directory not found at {args.output_dir}")
    return

checkpoint_path = os.path.join(args.output_dir, 'checkpoint-best.pth')
if not os.path.exists(checkpoint_path):
    print(f"Error: 'checkpoint-best.pth' not found in {args.output_dir}. Cannot evaluate model.")
    model_args_loaded = False
else:
    try:
        model_args = torch.load(checkpoint_path, map_location='cpu')['args']
        model_args.output_dir = args.output_dir
        model_args_loaded = True
    except Exception as e:
        print(f"Could not load args from checkpoint: {e}. Cannot evaluate or plot CIs.")
        model_args_loaded = False

log_path = os.path.join(args.output_dir, 'log.txt')
if os.path.exists(log_path):
    dataset_sizes = {}
    if model_args_loaded:
        try:
            train_ds, test_ds, val_ds, _, _ = get_dataset(model_args)
            dataset_sizes = {'train': len(train_ds), 'test': len(test_ds), 'val': len(val_ds)}
            print(f"Dataset sizes: Train={dataset_sizes['train']}, Val={dataset_sizes['val']}, Test={dataset_sizes['test']}")
        except Exception as e:
            print(f"Could not load datasets: {e}. Plotting without confidence intervals.")
    
    print("\nPlotting training curves...")
    plot_curves(log_path, args.output_dir, dataset_sizes)
else:
    print(f"log.txt not found in {args.output_dir}, skipping plotting.")